In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [8]:
# Load dataset — needs ISO-8859-1 encoding (not UTF-8)
df = pd.read_csv('../data/raw/online_retail_II.csv', encoding='ISO-8859-1')

In [15]:
print(f"Shape: {df.shape}")

Shape: (1067371, 9)


In [5]:
print(f"Columns: {list(df.columns)}")

Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [6]:
print(f"\nDtypes:\n{df.dtypes}")


Dtypes:
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [8]:
print(f"\nFirst 3 rows:")
df.head(3)


First 3 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


In [16]:
print(f"\nLast 3 rows:")
df.tail(3)


Last 3 rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,YearMonth
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,2011-12
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France,2011-12
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680.0,France,2011-12


In [9]:
print("=" * 55)
print("DATA QUALITY AUDIT")
print("=" * 55)

DATA QUALITY AUDIT


In [10]:
# 1. Shape and memory
print(f"\nRows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Memory:  {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")


Rows:    1,067,371
Columns: 8
Memory:  361.9 MB


In [12]:
# 2. Missing values
print("\n--- Missing Values ---")
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
for col in df.columns:
    if nulls[col] > 0:
        print(f"  {col}: {nulls[col]:,} ({null_pct[col]}%)")


--- Missing Values ---
  Description: 4,382 (0.41%)
  Customer ID: 243,007 (22.77%)


In [13]:
# 3. Cancellations (Invoice starts with 'C')
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"\n--- Cancellations ---")
print(f"  Cancelled invoices: {cancelled['Invoice'].nunique():,}")
print(f"  Cancelled rows:     {len(cancelled):,} ({len(cancelled)/len(df)*100:.1f}%)")



--- Cancellations ---
  Cancelled invoices: 8,292
  Cancelled rows:     19,494 (1.8%)


In [14]:
# 4. Negative quantities (should match cancellations)
neg_qty = df[df['Quantity'] < 0]
print(f"\n--- Negative Quantities ---")
print(f"  Rows with Quantity < 0: {len(neg_qty):,}")


--- Negative Quantities ---
  Rows with Quantity < 0: 22,950


In [15]:
# 5. Zero or negative price
bad_price = df[df['Price'] <= 0]
print(f"\n--- Invalid Prices ---")
print(f"  Rows with Price <= 0: {len(bad_price):,}")



--- Invalid Prices ---
  Rows with Price <= 0: 6,207


In [16]:
# 6. Duplicate rows
dupes = df.duplicated().sum()
print(f"\n--- Duplicate Rows ---")
print(f"  Full duplicates: {dupes:,}")


--- Duplicate Rows ---
  Full duplicates: 34,335


In [17]:
# 7. Date range
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(f"\n--- Date Range ---")
print(f"  Earliest: {df['InvoiceDate'].min()}")
print(f"  Latest:   {df['InvoiceDate'].max()}")


--- Date Range ---
  Earliest: 2009-12-01 07:45:00
  Latest:   2011-12-09 12:50:00


In [18]:
# 8. Countries
print(f"\n--- Geography ---")
print(f"  Unique countries: {df['Country'].nunique()}")
print(f"  Top 5:\n{df['Country'].value_counts().head()}")


--- Geography ---
  Unique countries: 43
  Top 5:
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Name: count, dtype: int64


In [19]:
# 9. Unique products and customers
print(f"\n--- Entities ---")
print(f"  Unique StockCodes:  {df['StockCode'].nunique():,}")
print(f"  Unique Customer IDs: {df['Customer ID'].nunique():,}")
print(f"  Unique Invoices:    {df['Invoice'].nunique():,}")


--- Entities ---
  Unique StockCodes:  5,305
  Unique Customer IDs: 5,942
  Unique Invoices:    53,628


In [9]:
from ydata_profiling import ProfileReport

# Generate full report (takes ~2 minutes for 1M rows)
profile = ProfileReport(
    df,
    title="Online Retail II — Data Profile",
    explorative=True,
    minimal=False
)

# Save HTML report
profile.to_file('../data/profiles/data_profile.html')
print("Profile saved to data/profiles/data_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile saved to data/profiles/data_profile.html


In [14]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Orders by country (top 10)
top_countries = df['Country'].value_counts().head(10)
axes[0].barh(top_countries.index[::-1], top_countries.values[::-1], color='#378ADD')
axes[0].set_title('Orders by country (top 10)')
axes[0].set_xlabel('Number of rows')

# 2. Monthly order volume
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('YearMonth').size()
axes[1].plot(monthly.index.astype(str), monthly.values, color='#1D9E75', linewidth=2)
axes[1].set_title('Monthly order volume')
axes[1].tick_params(axis='x', rotation=45)

# 3. Quantity distribution (excluding outliers)
clean_qty = df[(df['Quantity'] > 0) & (df['Quantity'] < 100)]['Quantity']
axes[2].hist(clean_qty, bins=40, color='#7F77DD', edgecolor='none')
axes[2].set_title('Quantity distribution (1–100)')
axes[2].set_xlabel('Quantity per line')

plt.tight_layout()
plt.savefig('../data/profiles/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved.")

Chart saved.
